# Train-Free Glioblastoma Segmentation in MRI with TDA
## II/III: Segmentation of fetal plate
Anton François & Raphaël Tinarrage

Data from https://dataverse.harvard.edu/dataset.xhtml?persistentId=doi:10.7910/DVN/WE9JVR

Paper https://www.nature.com/articles/s41598-017-00525-w

In [ ]:
# TDA
import persim, cripser

# Image
import scipy, skimage
import matplotlib as mpl

# Anton
from parse_brats_for_raph import *
import nibabel as nib
import vedo

# Misc
import itertools

# Comments
import sys, time, datetime, IPython.display

def ChronometerStart(msg='Start... '):
    start_time = time.time()
    sys.stdout.write(msg); sys.stdout.flush()
    return start_time
def ChronometerStop(start_time, method='ms', linebreak='\n'):
    elapsed_time_secs = time.time() - start_time
    if method == 'ms':
        msg = 'Execution time: '+repr(round(elapsed_time_secs*1000))+' ms.'+linebreak
    if method == 's':
        msg = 'Execution time: '+repr(round(elapsed_time_secs))+' s.'+linebreak
    sys.stdout.write(msg); sys.stdout.flush()    
def ChronometerTick(start_time, i, i_total, msg):    
    elapsed_time_secs = time.time() - start_time
    expected_time_secs = (i_total-i-1)/(i+1)*elapsed_time_secs
    msg1 = 'It '+repr(i+1)+'/'+repr(i_total)+'. '
    msg2 = 'Duration %s ' % datetime.timedelta(seconds=round(elapsed_time_secs))
    msg3 = 'Expected remaining time %s.' % datetime.timedelta(seconds=round(expected_time_secs))
    sys.stdout.write('\r'+msg+msg1+msg2+msg3)
    if i>=i_total-1: sys.stdout.write('\n')
        
''' FUNCTIONS TO PROCESS IMAGES '''

def argmax_image(img):
    return np.unravel_index(img.argmax(), img.shape)

def GetConnectedComponent(img, pos, t):
    '''
    Get the connected component of the voxel pos = (x,y,z) at time t.
    The output is a binary image.
    Background value of img must be 0 (as conventional).
    '''
    imt = (img>=t)*1
    if imt[pos[0],pos[1],pos[2]]==0: # The voxel pos is not active at time t
        CC = img*0
    else:    
        labels = skimage.measure.label(imt, background=0)
        labeltumor = labels[pos[0],pos[1],pos[2]]
        CC = (labels == labeltumor)*1
        if labeltumor==0: print('Problem! The label is background :(')
    return CC

def get_largest_CC(img, t, verbose=False):    
    if verbose: start_time = ChronometerStart('Get largest connected component... ')
    imgt = (img>=t)*1
    Labels = skimage.measure.label(imgt, background=0)
    nlabels = np.max(Labels)
    CardinalLabels = [0]+[np.sum(Labels==i) for i in range(1,nlabels)]
    ilabel = np.argmax(CardinalLabels)
    CC = (Labels==ilabel)*1    
    if verbose: ChronometerStop(start_time, method='s')
    return CC

def SEN(seg_1,seg_2, verbose=False):
    prod_seg = seg_1 * seg_2
    if seg_1.sum()==0: sen = 1
    else:              sen = prod_seg.sum() / seg_1.sum()
    return sen

def GetSensitivities(seg_1,seg_2, verbose=False):
    sens = {l:SEN((seg_1==l)*1, (seg_2==l)*1, verbose=False) for l in [1,2,4]}
    sens[0] = SEN((seg_1>0)*1, (seg_2>0)*1, verbose=False)
    return sens

def ACC(seg_1,seg_2, verbose=False):
    prod_seg = seg_1 * seg_2
    prod_seg_complement = (1-seg_1) * (1-seg_2)
    acc = (prod_seg.sum()+prod_seg_complement.sum())/np.prod(np.shape(seg_1))
    return acc

def GetAccuracies(seg_1,seg_2, verbose=False):
    accs = {l:ACC((seg_1==l)*1, (seg_2==l)*1, verbose=False) for l in [1,2,4]}
    accs[0] = ACC((seg_1>0)*1, (seg_2>0)*1, verbose=False)
    return accs

def DICE(seg_1,seg_2, verbose=False):
    prod_seg = seg_1 * seg_2
    sum_seg = seg_1 + seg_2
    if sum_seg.sum()==0: dice = 1
    else: dice = 2 * prod_seg.sum() / sum_seg.sum()
    if verbose:
        # Non-symmetric scores
        diceleft  = prod_seg.sum() / seg_1.sum()
        diceright = prod_seg.sum() / seg_2.sum()
        print('Sørensen–Dice coefficient: ',round(dice,3), round(diceleft,3), round(diceright,3))
    return dice

def GetDICEs(seg_1,seg_2, verbose=False):
    dices = {l:DICE((seg_1==l)*1, (seg_2==l)*1, verbose=False) for l in [1,2,4]}
    dices[0] = DICE((seg_1>0)*1, (seg_2>0)*1, verbose=False)
    if verbose: print('Sørensen–Dice coefficients:', {l:round(dices[l],3) for l in [1,2,4]})
    return dices 

def OpenBRATS(i,verbose=False):
    '''
    i must be between 0 and 1249 included
    '''
    indices_BRATS2021 = list(range(707))+list(range(708,1089))+list(range(1090,1099))+list(range(1100,1253))
        #images that I can open on my laptop
    if i<0 or 1>=1250:
        print('Error in OpenBRATS! i = ', i)
        return None
    else:    
        n_image = indices_BRATS2021[i]
        # Open t1ce
        pb = parse_brats(brats_list=None,brats_folder='2021',modality='t1ce')
        img_t1ce, _ = pb(n_image,to_torch=False)
        # Open flair
        pb = parse_brats(brats_list=None,brats_folder='2021',modality='flair')
        img_flair, seg_medecin = pb(n_image,to_torch=False)
        seg_medecin = seg_medecin.get_fdata()
        if verbose: print(pb.brats_list[n_image])
        return pb, img_flair, img_t1ce, seg_medecin
    
''' MAIN ALGORITHM '''

def Preprocess(img_flair,img_t1ce,sigma,normalize='max',do_enhance=[False,False]):
    '''
    do_enhance can be True, False, or a list [True,True], [True, False], ...
    '''
    # Enhance image
    mask = (img_flair!=0)
    if do_enhance==True:
        img_flair = skimage.filters.rank.enhance_contrast(img_flair,footprint=skimage.morphology.ball(4), mask=mask)
        img_t1ce = skimage.filters.rank.enhance_contrast(img_t1ce,footprint=skimage.morphology.ball(4), mask=mask)
    elif do_enhance is not False:
        if do_enhance[0]:
            img_flair = skimage.filters.rank.enhance_contrast(img_flair,footprint=skimage.morphology.ball(4), mask=mask)
        if do_enhance[1]:
            img_t1ce = skimage.filters.rank.enhance_contrast(img_t1ce,footprint=skimage.morphology.ball(4), mask=mask)

    # Normalize images
    if normalize=='max':
        img_flair = img_flair/np.max(img_flair)
        img_t1ce = img_t1ce/np.max(img_t1ce)
    else:
        img_flair = img_flair/255
        img_t1ce = img_t1ce/255
    
    # N4 Bias Field Correction
    # see https://simpleitk.readthedocs.io/en/master/link_N4BiasFieldCorrection_docs.html
    
    # Gaussian smoothing
    if sigma>0:
        img_flair = scipy.ndimage.gaussian_filter(img_flair, sigma=sigma)    
        img_t1ce = scipy.ndimage.gaussian_filter(img_t1ce, sigma=sigma)
            
    return img_flair, img_t1ce

def suggest_t(img, N=100, dt_threshold=1, seg_medecin=None, verbose=False, plot=False):
    T = np.linspace(0,1,N)

    # Build suggestion curve
    if verbose: start_time = ChronometerStart('Suggest t... ')
    S = np.array([np.sum(img>t) for t in T])
    S_dt = (S[:-1] - S[1:])                #finite differences
    S_dt_norm = S_dt*len(S_dt)/S_dt.sum()  #S_dt_norm has integral 1, i.e., np.sum(S_dt_norm)/len(S_dt) = 1
    if verbose: ChronometerStop(start_time, method='s')
        
    # Find optimal t
    best_i = np.where(S_dt_norm>dt_threshold)[0][-1] #last value for which S_dt_norm>dt_threshold
    best_t = T[best_i+1]

    if plot:
        fig, ax = plt.subplots(1, 1, figsize = (8,2))
        plt.plot(T,S/np.max(S)*np.max(S_dt_norm),c='black', label='_nolegend_')
        plt.plot(T[0:-1],S_dt_norm, c='black', label='_nolegend_')
        plt.scatter(T[0:-1],S_dt_norm,c='black',s=10, label='_nolegend_')
        plt.axhline(dt_threshold, c='blue', label='dt_threshold') 
        plt.axvline(best_t, c='pink', label='best t')
        if seg_medecin is not None:
            groundtruth_t = np.min(img[seg_medecin>0])
            plt.axvline(groundtruth_t, c='red', label='groundtruth t')
        plt.ylim(0,np.max(S_dt_norm[1::])*1.1)
        plt.title('Suggest t, dt_threshold = '+repr(dt_threshold)); plt.legend(); plt.show()

    return best_t

# def Step1(img_flair,seg_medecin=None,method='suggest_t',dt_threshold=1,verbose=True,plot=True):
#     '''
#     method can be 'suggest_t', 'medecin' or 'medecin_hull'
#     '''        
#     if method=='suggest_t':
#         # Find best t
#         t = suggest_t(img_flair, dt_threshold=dt_threshold, seg_medecin=seg_medecin, verbose=verbose, plot=plot)

#         # Extract CC
#         seg_union = get_largest_component(img_flair, t, verbose=verbose)

#         # Fill CC with holes
#         if verbose: start_time = ChronometerStart('Fill the holes... ')
#         seg_union = scipy.ndimage.binary_fill_holes(seg_union)
#         if verbose: ChronometerStop(start_time, method='s')
            
#         # Smooth via binary closing
#         iterations = 0
#         if iterations>0: seg_union = scipy.ndimage.morphology.binary_closing(seg_union, iterations=iterations)

#     elif method=='medecin': seg_union = (seg_medecin>0)*1

#     elif method=='medecin_hull': 
#         # Take convex hull of seg_union
#         seg_union = (seg_medecin>0)*1
#         if verbose: start_time = ChronometerStart('Take convex hull... ')
#         seg_union = (seg_medecin>0)*1
#         seg_union = skimage.morphology.convex_hull_image(seg_union)
#         seg_union = seg_union*1
#         if verbose: ChronometerStop(start_time, method='s')

#     return seg_union

# def Step2(img_t1ce,seg_union,H2_features_max=5,verbose=True,plot=True):
#     # Compute PH2
#     if verbose: start_time = ChronometerStart('Compute diagram... ')
#     seg_union_t1ce = img_t1ce*seg_union
#     barcode = cripser.computePH(1-seg_union_t1ce,maxdim=3) # Compute diagram
#     H2 = [list(bar[1::]) for bar in barcode if bar[0]==2 and bar[2]<1] # Only non-infinite bars
#     H2 = [bar for _,bar in sorted(zip([bar[1]-bar[0] for bar in H2],H2))[::-1]] # Sort list H2 by persistence
#     if verbose: ChronometerStop(start_time, method='s')

#     # If empty diagram
#     if len(H2)==0: return seg_union_t1ce*0
        
#     # Plot diagram
#     if plot:
#         fig, ax = plt.subplots(1,1, figsize=(4,4))
#         persim.plot_diagrams([np.array([barcode[0][1:3]]), np.array([barcode[0][1:3]]), np.array([bar[1:3] for bar in barcode if bar[0]==3-1])])
#         plt.title('Persistence diagram of t1ce segmented',fontsize=10)

#     # Compute width of the holes of H2-features
#     i_max = min(H2_features_max,len(H2)) #number of top bars to parse
#     length_holes = []
#     for i in range(i_max):
#         bar = H2[i]
#         pos = np.array(bar[2:5]).astype(int)
#         t = bar[0]+0.0001
#         seg_contour = GetConnectedComponent(seg_union_t1ce, pos, 1-t)
#         seg_contour_filled = scipy.ndimage.binary_fill_holes(seg_contour)
#         length_holes.append( np.sum(seg_contour_filled-seg_contour) )
#     if verbose: print('Width of the holes:', {i:length_holes[i] for i in range(i_max)})
    
#     # Select the largest hole
#     i = np.argmax(length_holes)
#     bar = H2[i]
#     pos = np.array(bar[2:5]).astype(int)
#     t = bar[0]+0.0001
#     seg_contour = GetConnectedComponent(seg_union_t1ce, pos, 1-t)

#     # Plot
#     if plot: patch = plt.Circle((bar[0],bar[1]), 0.01,fill=False); ax.add_patch(patch)
#     return seg_contour

# def Step3(seg_union, seg_contour, alpha_boundary=.85,verbose=True):               
#     # 3rd step: - Classify components
#         # 1 - RED, TC  -> NECROSE INACTIVE, TUMORUS CORE
#         # 2 - BLUE, ED -> INFILTRATION, OEDEME
#         # 4 - ORANGE, ET -> NECROSE ACTIVE, ENHANCING TUMOR
#     seg_union_nocontour = seg_union - seg_contour
#     seg_nocontour_components = []
#     labels = skimage.measure.label(seg_union_nocontour, background=0)

#     N = 0 # select components with cardinal at least N
#     components = [(labels==i)*1 for i in range(1,np.max(labels)+1)]
#     components = [component for component in components if np.sum(component)>N]

#     # Classify components: TC or WT
#     seg_final = seg_contour.copy()*4 # define seg_final
#     for component in components:
#         componentdilated = scipy.ndimage.binary_dilation(component,iterations=1)
#         componentcontour = componentdilated - component
#         meanvalue = np.mean(seg_contour[np.where(componentcontour>0)])
#         if meanvalue<alpha_boundary: #sort du masque Segmentation, label 2
#             seg_final[component>0] = 2
#         else: #label 1
#             seg_final[component>0] = 1
            
#     return seg_final

''' FUNCTIONS TO PLOT '''

from matplotlib.colors import ListedColormap

# Own colormap for segmentations
cmap_segs = ListedColormap([[0,0,0,0],'tab:red','tab:blue','tab:orange'])
DLT_KW_IMAGE = dict(cmap='gray',origin='lower',vmin=0,vmax=1)
DLT_KW_SEG= dict(cmap=cmap_segs,interpolation='nearest',origin='lower')
                 
def make_3d_flat(img_3D,pos):
    ''' 
    Take a 'brain' 3D image, take 3 slices and make a long 2D image of it.
    '''
    crop = 2 #parameter
    # Cut slices
    D,H,W = img_3D.shape
    im0 = img_3D[:,:,pos[2]].T #image_slice(img_3D,slice[2],2).T
    im1 = img_3D[:,pos[1],:].T#image_slice(img_3D,slice[1],1).T
    im2 = img_3D[pos[0],:,:].T#image_slice(img_3D,slice[0],0).T
    # Crop and pad slices
    long_img = np.zeros((D,D+H+H-int(3.5*crop)))
    long_img[:D,:D-crop] = im0[:,crop//2:-crop//2]
    long_img[(D-W)//2:(D-W)//2 + W,D-int(1.7*crop):D+H-int(2.7*crop)] = im1[::-1,crop//2:-crop//2]
    long_img[(D-W)//2:(D-W)//2 + W,D+H-int(3*crop):] = im2[::-1,crop//2:]
    return long_img

def PlotSegmentation(pb,img,seg=None,pos=None,figsize=(15,5)):
    if pos is None: pos = argmax_image(img)
    fig = plt.figure(figsize=figsize); ax = fig.add_subplot(1,1,1)
    ax.imshow(make_3d_flat(img,pos),cmap='gray',alpha=0.5,origin='upper')
    if seg is None: seg = img*0
    ax.imshow(make_3d_flat(seg,pos),**DLT_KW_SEG)
    ax.text(235,15,pb.brats_list[i],c='white',fontsize=20)
    plt.axis('off')
    
def PlotSuperlevelSets(image, pos=None, Times=None):
    if Times==None: Times = [0,0.05,0.3,0.5,1]

    if pos==None: 
        if image[0,0,0]==0: pos = argmax_image(image)
        else: pos = argmax_image(1-image)
    im = image[:,:,pos[2]]

    fig = plt.figure(figsize=(15,15)) # Notice the equal aspect ratio
    ax = [fig.add_subplot(1,len(Times),i+1) for i in range(len(Times))]
    ax.reverse()
    for i in range(len(Times)):
        t = Times[i]
        imt = np.zeros(np.shape(im)); imt[im<t] = 1; imt[im>=t] = 0
        ax[i].imshow(imt,origin='lower',vmin=0,vmax=1,cmap='gray')
        ax[i].set_title('t = '+repr(Times[::-1][i]), fontsize=17.5); 
        ax[i].axis('off'); ax[i].set_aspect('equal')
    fig.subplots_adjust(wspace=0.05, hspace=0)
    
def PlotSublevelSets(image, pos=None, Times=None):
    if Times==None: Times = [0,0.05,0.3,0.5,1]

    if pos==None: 
        if image[0,0,0]==0: pos = argmax_image(image)
        else: pos = argmax_image(1-image)
    im = image[:,:,pos[2]]

    fig = plt.figure(figsize=(15,15)) # Notice the equal aspect ratio
    ax = [fig.add_subplot(1,len(Times),i+1) for i in range(len(Times))]
    for i in range(len(Times)):
        t = Times[i]
        imt = np.zeros(np.shape(im)); imt[im>t] = 1; imt[im<=t] = 0
        ax[i].imshow(imt,origin='lower',vmin=0,vmax=1,cmap='gray')
        ax[i].set_title('t = '+repr(t), fontsize=17.5); 
        ax[i].axis('off'); ax[i].set_aspect('equal')
    fig.subplots_adjust(wspace=0.05, hspace=0)    
    
def BoxplotSegmentation(dic, suptitle=None):
    Titles = ['WT', 'TC', 'ED', 'ET']
    Labels = [0,1,2,4]
    DICEs_list = list(dic.values())

    plt.figure(figsize =(4*4, 2),constrained_layout=True)
    for k in range(len(Labels)):
        label = Labels[k]
        scores = [dic[i][label] for i in dic]
        ax = plt.subplot(1,len(Labels),k+1)
        ax.boxplot(scores, showmeans=True, notch = True)
        title = Titles[k]+', mean='+repr(round(np.mean(scores),2))+', med='+repr(round(np.median(scores),2))
        ax.title.set_text(title)
        ax.set_ylim(-0.01,1)
    if suptitle is not None: plt.suptitle(suptitle)

In [ ]:
def PlotSuperlevelSets(image, pos=None, Times=None):
    if Times==None: Times = [0,0.05,0.3,0.5,1]

    if pos==None: 
        if image[0,0,0]==0: pos = argmax_image(image)
        else: pos = argmax_image(1-image)
    im = image[:,:,pos[2]]

    fig = plt.figure(figsize=(15,15)) # Notice the equal aspect ratio
    ax = [fig.add_subplot(1,len(Times),i+1) for i in range(len(Times))]
    for i in range(len(Times)):
        t = Times[i]
        imt = np.zeros(np.shape(im)); imt[im<t] = 1; imt[im>=t] = 0
        ax[i].imshow(imt,origin='lower',vmin=0,vmax=1,cmap='gray')
        ax[i].set_title('t = '+repr(Times[i]), fontsize=17.5); 
        ax[i].axis('off'); ax[i].set_aspect('equal')
    fig.subplots_adjust(wspace=0.05, hspace=0)

In [ ]:
def OpenSAT(n_image, verbose=False):
    ' i must be between 21 and 38 included '
    # Open image
    if n_image <= 35: filename = '/home/raph/GoogleDrive/Professionnel/TDA+Brains/Code Python/data/fetal/STA'+str(n_image)+'.nii.gz'
    else:             filename = '/home/raph/GoogleDrive/Professionnel/TDA+Brains/Code Python/data/fetal/STA'+str(n_image)+'exp.nii.gz'
    img = nib_load(filename).get_fdata();
    img /= np.max(img)

    # Open tissue
    if n_image <= 35: filename = '/home/raph/GoogleDrive/Professionnel/TDA+Brains/Code Python/data/fetal/STA'+str(n_image)+'_tissue.nii.gz'
    else:             filename = '/home/raph/GoogleDrive/Professionnel/TDA+Brains/Code Python/data/fetal/STA'+str(n_image)+'exp_tissue.nii.gz'
    img_tissue = nib_load(filename).get_fdata(); 

    # Define segmentation cortical plate
    img_seg = ((img_tissue==112)+(img_tissue==113))*1
#     img_seg = ((img_tissue==112)+(img_tissue==113)+(img_tissue==114)+(img_tissue==115))*1
#     img_seg = ((img_tissue>=120)*(img_tissue<=125))*1
    if verbose: print('Segmentation size:', np.sum(img_seg))

    return img, img_seg

In [ ]:
def Step2_SAT(img,min_dist_bars=0.003,ratio_upper=3/4,ratio_lower=1/100,verbose=False,plot=False,\
             select_bar='birth',y_im=None):
    seg_final = np.zeros(np.shape(img))
    
    # Get nonempty slices of im
    if y_im == None:
        nonempty_slices = np.where(np.sum(np.sum(1-img,0),1))[0]
        ymin_im, ymax_im = nonempty_slices[0], nonempty_slices[-1]
        y_im = range(ymin_im,ymax_im+1)
        if verbose: print('ymin ymax =', ymin_im, ymax_im)

    DICEbySLICE = dict()

    msg = 'Compute DICEs... '
    if not verbose: start_time = ChronometerStart(msg)
    for y in y_im:
        ' Select only one bar '

        # Define slice
        img_slice = img[:,y,:].copy()
        seg_medecin_slice = seg_medecin[:,y,:].copy()

        # Compute barcode
        if verbose: start_time = ChronometerStart(repr(y)+' Compute diagram... ')
        barcode = cripser.computePH(img_slice,maxdim=1)        # Compute diagram
        if verbose: ChronometerStop(start_time, method='s')

        # Sort barcode by longest bars
        homology_dim = 1
        H = [list(bar[1::]) for bar in barcode if bar[0]==homology_dim]          # Allow infinite bars
        H = [bar for _,bar in sorted(zip([bar[1]-bar[0] for bar in H],H))[::-1]] # Sort list H by persistence

        # Discard bars
        H_CC, H_width_hole_interior = dict(), dict()
        for bar in H:
            # Get CC
            pos, t = np.array(bar[2:4]).astype(int), bar[0]+0.0000001
            labels = skimage.measure.label((img_slice<=t)*1, background=0)
            CC = (labels == labels[pos[0],pos[1]])*1

            # Compute width hole
            labels = skimage.measure.label(1 - CC, background=0)
            components = [(labels==i)*1 for i in range(1,np.max(labels)+1)]
            components_len = [np.sum(component) for component in components]
            width_hole_interior = np.size(img_slice) - max(components_len) - np.sum(CC) 
                        # width of the hole, without the boundary
            width_hole = np.size(img_slice) - max(components_len)              
                        # width of the hole, with the boundary

            # Discard (1): CC must not touch the boundary
            CC_filled      = scipy.ndimage.binary_fill_holes(CC).astype(int)
            boundary       = scipy.ndimage.binary_dilation(CC_filled,iterations=1)-CC_filled
            value_boundary = np.mean(img_slice[np.where(boundary)])
            if value_boundary >= 1: continue

            # Discard (2): the whole must not be too large or too small
            size_slice, size_CC = np.sum(img_slice<1), np.sum(CC)
            if size_CC > ratio_upper*size_slice or \
               width_hole_interior < ratio_lower*size_slice: continue  

            # Save
            H_CC[tuple(bar)], H_width_hole_interior[tuple(bar)] = CC, width_hole_interior

        if verbose: print('Saved/discarded bars:', len(H), '--->', len(H_CC))

        # Plot diagram
        if plot and len(H)>=1:
            fig = plt.figure(figsize=(12,2)); ax = fig.add_subplot(1,4,1)
            dim_max = int(max([bar[0] for bar in barcode]))
            persim.plot_diagrams([np.array([bar[1:3] for bar in barcode if bar[0]==i]) for i in range(dim_max+1)],ax=ax)
            ax.set_title('Persistence diagram of flair',fontsize=10); ax.set_xlim([0,1.2]); ax.set_ylim([0,1.2]);

            if len(H_CC)>=1:
                ax = fig.add_subplot(1,4,2)
                persim.plot_diagrams([np.array([ [0,0] ]), np.array([bar[1:3] for bar in barcode if tuple(bar[1::]) in H_CC])],ax=ax)
                ax.set_title('Relevant diagram',fontsize=10); ax.set_xlim([0,1.2]); ax.set_ylim([0,1.2]);

        # Get optimal bar
        if len(H_CC)>=1:
            # Bar with max width
            if select_bar=='width': 
                bar = max(H_width_hole_interior, key=H_width_hole_interior.get)

            # Bar with max pers
            elif select_bar=='pers':
                H_pers = {bar:(bar[1]-bar[0]) for bar in H_CC}
                bar = max(H_pers, key=H_pers.get)            

            # Bar with min birth
            elif select_bar=='birth':
                H_birth = {bar:bar[0] for bar in H_CC}
                bar = min(H_birth, key=H_birth.get)            

            CC = H_CC[bar]
            if plot: patch = plt.Circle((bar[0],bar[1]), 0.025,fill=False); ax.add_patch(patch)

            # Check if exists a similar bar
            if len(H_CC)>=2:
                distance_bars = { bar2:np.linalg.norm(np.array(bar[0:2])-np.array(bar2[0:2])) for bar2 in H_CC }
                del distance_bars[bar]
                bar_closest = min(distance_bars, key=distance_bars.get)
                dist = distance_bars[bar_closest]
                if verbose: 
                    if dist>min_dist_bars:    print('No similar bar:', dist, '>', min_dist_bars)
                    elif dist<=min_dist_bars: print('Exists a similar bar:', dist, '<=', min_dist_bars)
                if dist<=min_dist_bars:
                    # Get CC
                    pos, t = np.array(bar_closest[2:4]).astype(int), bar_closest[0]+0.0000001
                    CC_closest = H_CC[bar_closest]
                    if plot: patch = plt.Circle((bar_closest[0],bar_closest[1]), 0.025,fill=False); ax.add_patch(patch)

                    # Join CC
                    CC = (CC+CC_closest>0)*1

        else: CC = np.zeros(np.shape(img_slice))

        # Compute dice and save CC
        DICEbySLICE[y] = DICE(CC,seg_medecin_slice, verbose=False)
        seg_final[:,y,:] = CC

        if verbose: print(y, 'Dice score ', round(DICEbySLICE[y],3), '<-------------------------------------------------------------------------')

        # Plot
        if plot:
            ax = fig.add_subplot(1,4,3); ax.axis('off'); ax.set_title('TDA seg');
            ax.imshow(img_slice,cmap='gray',alpha=0.5,origin='lower'); 
            ax.imshow(CC,alpha=0.5,**DLT_KW_SEG); 

            ax = fig.add_subplot(1,4,4); ax.axis('off'); ax.set_title('Groundtruth');
            ax.imshow(img_slice,cmap='gray',alpha=0.5,origin='lower'); 
            ax.imshow(seg_medecin_slice,alpha=0.5,**DLT_KW_SEG);
            plt.show()

        if not verbose: ChronometerTick(start_time, y-ymin_im, ymax_im-ymin_im+1, msg)
            
    return seg_final, DICEbySLICE

# I - SAT dataset

# Plot images

In [ ]:
' Figure article - coronal '

# for n_image in range(21,38+1):
z_pos = [int(i) for i in np.linspace(60,np.shape(img)[1]-60,7)]
z_pos = [int(i) for i in np.linspace(60,np.shape(img)[1]-70,5)]
figsize=(len(z_pos)*5,3*5)
fig = plt.figure(figsize=figsize); 
fig.subplots_adjust(wspace=0.05, hspace=0.05)

for i_image, n_image in enumerate([21,30,38]):
    # Open image
    img, img_seg = OpenSAT(n_image)

    # Plot slices
    for i in range(len(z_pos)):
        ax = fig.add_subplot(3,len(z_pos),i+1+i_image*len(z_pos)); ax.axis('off')
        ax.imshow(img[:,z_pos[i],10:145].T,cmap='gray',alpha=1,origin='lower'); 
        ax.imshow(img_seg[:,z_pos[i],10:145].T,alpha=0.7,**DLT_KW_SEG); 
        
# plt.show()
plt.savefig('Images/fetal_examples.pdf', format="pdf", bbox_inches="tight")

In [ ]:
' Plot only one image - horizontal '

n_image = 30 #between 21 and 38 included

# Open image
img, img_seg = OpenSAT(n_image)
print('n_image =', n_image, '/ shape =', np.shape(img))

# Plot slices
z_pos = [60,93,105]
# z_pos = [int(i) for i in np.linspace(60,np.shape(img)[2]-50,5)]
figsize=(len(z_pos)*5,5)
fig = plt.figure(figsize=figsize); 
fig.subplots_adjust(wspace=0.05, hspace=0)
for i in range(len(z_pos)):
    ax = fig.add_subplot(1,len(z_pos),i+1); ax.axis('off')
    ax.imshow(img[:,:,z_pos[i]],cmap='gray',alpha=0.9,origin='lower'); 
    ax.imshow(img_seg[:,:,z_pos[i]],alpha=0.5,**DLT_KW_SEG); 
ax.set_title('n_image = '+repr(n_image));

In [ ]:
' Plot only one image - coronal '

n_image = 30 #between 21 and 38 included

# Open image
img, img_seg = OpenSAT(n_image)
print('n_image =', n_image, '/ shape =', np.shape(img))

# Plot slices
z_pos = [79,139]
# z_pos = [int(i) for i in np.linspace(60,np.shape(img)[1]-50,5)]
figsize=(len(z_pos)*5,5)
fig = plt.figure(figsize=figsize); 
fig.subplots_adjust(wspace=0.05, hspace=0)
for i in range(len(z_pos)):
    ax = fig.add_subplot(1,len(z_pos),i+1); ax.axis('off')
    ax.imshow(img[:,z_pos[i],:],cmap='gray',alpha=0.9,origin='lower'); 
    ax.imshow(img_seg[:,z_pos[i],:],alpha=0.5,**DLT_KW_SEG); 
ax.set_title('GW 30, coronal slices 79 and 139, n_image = '+repr(n_image));

# Save for Anton
plt.savefig('Images/Slices_Fetal.pdf', format="pdf", bbox_inches="tight")

In [ ]:
' Save pdf for Anton '

verbose, plot = True, True
min_dist_bars = 0.03

n_image = 30 #between 21 and 38 included

# Parameters
normalize= 'max'         # Preprocess, divide by max or 255
sigma = 0.1                # Preprocess, Gaussian blur
# dt_threshold = 1         # Step 1, threshold for suggest_t
# H_features_max = 5      # Step 2, number of H2 bars to consider
radius_ball = 0          # Step 2, dilation parameter
min_dist_bars = 0.03     # Step 2, to consider multiple bars
zero_boundary = 0.3      # Preprocess, to suppress the boundary

# Open image
img, seg_medecin = OpenSAT(n_image)
seg_final = np.zeros(np.shape(img))
print('n_image =', n_image, '/ shape =', np.shape(img))

# Preprocess
if radius_ball>0:   img = skimage.morphology.erosion(img, footprint=skimage.morphology.ball(radius_ball), out=None, shift_x=False, shift_y=False)
if sigma>0:         img = scipy.ndimage.gaussian_filter(img, sigma=sigma)    
if zero_boundary>0: img[np.where(img<zero_boundary)]=1

for y in z_pos:
    # Define slice
    img_slice = img[:,y,:].copy()
    seg_medecin_slice = seg_medecin[:,y,:].copy()

    # Compute barcode
    if verbose: start_time = ChronometerStart(repr(y)+' Compute diagram... ')
    barcode = cripser.computePH(img_slice,maxdim=1) # Compute diagram
    if verbose: ChronometerStop(start_time, method='s')

    # Get longest bar
    homology_dim = 1
    H = [list(bar[1::]) for bar in barcode if bar[0]==homology_dim]          # Allow infinite bars
    H = [bar for _,bar in sorted(zip([bar[1]-bar[0] for bar in H],H))[::-1]] # Sort list H by persistence

    if len(H)>=1:
        # Plot diagram
        if plot:
            fig = plt.figure(figsize=(4,4)); ax = fig.add_subplot(1,1,1)
            dim_max = int(max([bar[0] for bar in barcode]))
            persim.plot_diagrams([np.array([bar[1:3] for bar in barcode if bar[0]==i]) for i in range(dim_max+1)],ax=ax)
            ax.set_title('Persistence diagram of flair',fontsize=10);

        # Get width holes
        WidthHoles = []
        for bar in H:
            # Get CC
            pos, t = np.array(bar[2:4]).astype(int), bar[0]+0.0000001
            labels = skimage.measure.label((img_slice<=t)*1, background=0)
            CC = (labels == labels[pos[0],pos[1]])*1

            # Compute width hole
            labels = skimage.measure.label(1 - CC, background=0)
            components = [(labels==i)*1 for i in range(1,np.max(labels)+1)]
            components_len = [np.sum(component) for component in components]
            width_hole = np.size(CC) - max(components_len) - np.sum(CC)
                # size of the image, minus the backgroud component, minus the CC
                
            if width_hole + np.sum(CC) < np.sum(img_slice<1) and width_hole > np.sum(img_slice<1)/10:       
                                # The cycle is not the entire boundary, and the width is not too small
                WidthHoles.append(width_hole)
            else:   
                WidthHoles.append(0)

        if any(WidthHoles): # If there is a nonzero value
            # Get optimal bar
            i = np.argsort(WidthHoles)[-1] # Cycle that bounds the largest hole
            bar = H[i]

            # Get CC
            pos, t = np.array(bar[2:4]).astype(int), bar[0]+0.0000001
            imt = (img_slice<=t)*1
            labels = skimage.measure.label(imt, background=0)
            labeltumor = labels[pos[0],pos[1]]
            CC = (labels == labeltumor)*1
            if plot: patch = plt.Circle((bar[0],bar[1]), 0.01,fill=False); ax.add_patch(patch)

            # Check if exists a similar bar
            if len(H)>=1:
                distance_bars = np.linalg.norm(np.array(H)[:,0:2]-bar[0:2],axis=1)
                j = np.argsort(distance_bars)[1]
                bar_closest = H[j] # Closest bar
                dist = np.linalg.norm(np.array(bar[0:2])-np.array(bar_closest[0:2]))
                if verbose: 
                    if dist>min_dist_bars:   print('No similar bar:', dist, '>', min_dist_bars)
                    elif dist<=min_dist_bars: print('Exists a similar bar:', dist, '<=', min_dist_bars)
                if dist<=min_dist_bars:
                    # Get CC
                    pos, t = np.array(bar_closest[2:4]).astype(int), bar_closest[0]+0.0000001
                    imt = (img_slice<=t)*1
                    labels = skimage.measure.label(imt, background=0)
                    labeltumor = labels[pos[0],pos[1]]
                    CC_closest = (labels == labeltumor)*1
                    if plot: patch = plt.Circle((bar_closest[0],bar_closest[1]), 0.01,fill=False); ax.add_patch(patch)

                    # Join CC
                    CC = (CC+CC_closest>0)*1
        else:
            CC = np.zeros(np.shape(img_slice))
            
        plt.title('Fetal - GW 30 - coronal slice '+repr(y))
        plt.savefig('Images/Diagram_Fetal_'+repr(y)+'.pdf', format="pdf", bbox_inches="tight")

#     # Plot
#     if plot:
#         ax = fig.add_subplot(1,3,2); ax.axis('off'); ax.set_title('TDA seg');
#         ax.imshow(img_slice,cmap='gray',alpha=0.5,origin='lower'); 
#         ax.imshow(CC,alpha=0.5,**DLT_KW_SEG); 

#         ax = fig.add_subplot(1,3,3); ax.axis('off'); ax.set_title('Groundtruth');
#         ax.imshow(img_slice,cmap='gray',alpha=0.5,origin='lower'); 
#         ax.imshow(seg_medecin_slice,alpha=0.5,**DLT_KW_SEG);
#         plt.show()

#     if verbose:
#         # Compute dice
#         dice = dice(CC,seg_medecin_slice, verbose=False)
#         print(y, 'Dice score ', round(dice,3), '<------------------------------------------------')


All images have shape (135, 189, 155)

In [ ]:
' Plot all images - z (horizontal plane) '

for n_image in range(21,38+1):
    # Open image
    img, img_seg = OpenSAT(n_image)

    # Plot slices
    z_pos = [int(i) for i in np.linspace(60,np.shape(img)[2]-60,4)]
    figsize=(len(z_pos)*5,5)
    fig = plt.figure(figsize=figsize); 
    fig.subplots_adjust(wspace=0.05, hspace=0)
    for i in range(len(z_pos)):
        ax = fig.add_subplot(1,len(z_pos),i+1); ax.axis('off')
        ax.imshow(img[:,:,z_pos[i]],cmap='gray',alpha=0.9,origin='lower'); 
        ax.imshow(img_seg[:,:,z_pos[i]],alpha=0.5,**DLT_KW_SEG); 
    ax.set_title('n_image = '+repr(n_image));
    plt.show()

In [ ]:
' Plot all images - y (coronal plane) '

for n_image in range(21,38+1):
    # Open image
    img, img_seg = OpenSAT(n_image)

    # Plot slices
    z_pos = [int(i) for i in np.linspace(60,np.shape(img)[1]-60,7)]
    figsize=(len(z_pos)*5,5)
    fig = plt.figure(figsize=figsize); 
    fig.subplots_adjust(wspace=0.05, hspace=0)
    for i in range(len(z_pos)):
        ax = fig.add_subplot(1,len(z_pos),i+1); ax.axis('off')
        ax.imshow(img[:,z_pos[i],:],cmap='gray',alpha=0.9,origin='lower'); 
        ax.imshow(img_seg[:,z_pos[i],:],alpha=0.5,**DLT_KW_SEG); 
    ax.set_title('n_image = '+repr(n_image));
    plt.show()

In [ ]:
' Plot all images - x (sagittal plane) '

for n_image in range(21,38+1):
    # Open image
    img, img_seg = OpenSAT(n_image)

    # Plot slices
    z_pos = [int(i) for i in np.linspace(60,np.shape(img)[0]-50,5)]
    figsize=(len(z_pos)*3,4)
    fig = plt.figure(figsize=figsize); 
    fig.subplots_adjust(wspace=0.05, hspace=0)
    for i in range(len(z_pos)):
        ax = fig.add_subplot(1,len(z_pos),i+1); ax.axis('off')
        ax.imshow(img[z_pos[i],:,:],cmap='gray',alpha=0.9,origin='lower'); 
        ax.imshow(img_seg[z_pos[i],:,:],alpha=0.5,**DLT_KW_SEG); 
    ax.set_title('n_image = '+repr(n_image));
    plt.show()

# Segmentation 2D

Stategies:
- weight cycles with width of hole 
To do:
- améliorer la condition on width
- analyser méticuleusement les gestation week 21, 22, 23, 24 & 27, 28

### Example on one image

In [ ]:
' Segmentation SAT - one image - all slices '

n_image = 22 #between 21 and 38 included

verbose, plot = False, False

# Parameters
normalize= 'max'           # Preprocess, divide by max or 255
sigma = 0.5                # Preprocess, Gaussian blur
radius_ball = 1            # Preprocess, dilation parameter
zero_boundary = 0.3        # Preprocess, to suppress the boundary
min_dist_bars = 0.03       # Step 2, to consider multiple bars
ratio_upper = 3/4          # Step 2, max relative size of object (circle)
ratio_lower = 1/50         # Step 2, min relative size of interior
select_bar = 'birth'       # Step 2, optimal bar according to 'width' (of hole), 'pers' (of bar) or 'birth' 
Parameters = [normalize,sigma,radius_ball,zero_boundary,min_dist_bars,ratio_upper,ratio_lower,select_bar]

# Open image
img, seg_medecin = OpenSAT(n_image)

# Preprocess
if radius_ball>0:   img = skimage.morphology.erosion(img, footprint=skimage.morphology.ball(radius_ball), out=None, shift_x=False, shift_y=False)
if sigma>0:         img = scipy.ndimage.gaussian_filter(img, sigma=sigma)    
if zero_boundary>0: img[np.where(img<zero_boundary)]=1
    
# Segmentation
seg_final, DICEbySLICE = Step2_SAT(img,min_dist_bars=min_dist_bars,ratio_upper=ratio_upper,ratio_lower=ratio_lower,select_bar=select_bar,verbose=verbose,plot=plot)

# Compute dice
dice = DICE(seg_final,seg_medecin, verbose=False);
print('Final Dice score ', round(dice,3));

# Plot Dice slice by slice
print('Mean dice slice by slice:', round(np.mean(list(DICEbySLICE.values())),3), '- parameters:', Parameters)
plt.figure(figsize=(10,2)); plt.title('Dice score slice by slice (non-empty slices only)')
plt.scatter(DICEbySLICE.keys(),DICEbySLICE.values(), c='black',s=8)
plt.plot(DICEbySLICE.keys(),DICEbySLICE.values(), c='black'); plt.show()

In [ ]:
' Segmentation SAT - one image - a few slices '

verbose, plot = True, True
y_im = range(70,74)

# Open image
img, seg_medecin = OpenSAT(n_image)

# Preprocess
if radius_ball>0:   img = skimage.morphology.erosion(img, footprint=skimage.morphology.ball(radius_ball), out=None, shift_x=False, shift_y=False)
if sigma>0:         img = scipy.ndimage.gaussian_filter(img, sigma=sigma)    
if zero_boundary>0: img[np.where(img<zero_boundary)]=1

# Segmentation
_, _ = Step2_SAT(img,min_dist_bars=min_dist_bars,ratio_upper=ratio_upper,ratio_lower=ratio_lower,select_bar=select_bar,y_im=y_im,verbose=verbose,plot=plot)

### Batch

In [ ]:
' Batch - one set of parameters '

verbose, plot = False, False

# Parameters
normalize= 'max'           # Preprocess, divide by max or 255
sigma = 0.5                # Preprocess, Gaussian blur
radius_ball = 1            # Preprocess, dilation parameter
zero_boundary = 0.3        # Preprocess, to suppress the boundary
min_dist_bars = 0.03       # Step 2, to consider multiple bars
ratio_upper = 3/4          # Step 2, max relative size of object (circle)
ratio_lower = 1/50         # Step 2, min relative size of interior
select_bar = 'birth'       # Step 2, optimal bar according to 'width' (of hole), 'pers' (of bar) or 'birth' 
Parameters = [normalize,sigma,radius_ball,zero_boundary,min_dist_bars,ratio_upper,ratio_lower,select_bar]

dices_batch = dict()
for n_image in range(21,38+1):
    IPython.display.clear_output(wait=True)
    print('n_image =', n_image)
    
    # Open image
    img, seg_medecin = OpenSAT(n_image)

    # Preprocess
    if radius_ball>0:   img = skimage.morphology.erosion(img, footprint=skimage.morphology.ball(radius_ball), out=None, shift_x=False, shift_y=False)
    if sigma>0:         img = scipy.ndimage.gaussian_filter(img, sigma=sigma)    
    if zero_boundary>0: img[np.where(img<zero_boundary)]=1

    # Segmentation
    seg_final, DICEbySLICE = Step2_SAT(img,min_dist_bars=min_dist_bars,ratio_upper=ratio_upper,ratio_lower=ratio_lower,select_bar=select_bar,verbose=verbose,plot=plot)

    # Save dice
    dices_batch[n_image] = DICE(seg_final,seg_medecin, verbose=False)

# Plot curve
eps = 0.03
plt.figure(figsize=(10,1.5))
plt.plot(dices_batch.keys(),dices_batch.values(), 'black')
plt.scatter(dices_batch.keys(),dices_batch.values(), c='black')
plt.xlim(21-0.5,38+0.5); plt.xticks(range(21,38+1)); 
plt.ylim(min(dices_batch.values())-eps,max(dices_batch.values())+eps); plt.yticks(np.linspace(min(dices_batch.values()),max(dices_batch.values()),5));
plt.title('Dice score per gestational week - Parameters: '+repr(Parameters));

# Mean dice score
print('Mean dice score:', round(np.mean(list(dices_batch.values())),4), '- parameters', Parameters)

In [ ]:
# Plot curve for article
eps = 0.03
plt.figure(figsize=(6,1.5))
plt.plot(dices_batch.keys(),dices_batch.values(), 'black')
plt.scatter(dices_batch.keys(),dices_batch.values(), c='black')
plt.xlim(21-0.5,38+0.5); plt.xticks(range(21,38+1,2)); 
plt.ylim(min(dices_batch.values())-eps,max(dices_batch.values())+eps); 
plt.yticks([0.4,0.6,0.8])
plt.xlabel('Gestational week',fontsize=12); plt.ylabel('Dice',fontsize=12)
# plt.title('Dice score per gestational week');
plt.savefig('results/scores_SAT_article.pdf',bbox_inches='tight')  

In [ ]:
' Batch - several sets of parameters '

verbose, plot = False, False

# Parameters
ParametersToTest = [ 
                    ['max',0.5,1,0.3,0.1,3/4,1/50,'birth'],\
                    ['max',0.5,1,0.3,0.075,3/4,1/50,'birth'],\
                    ['max',0.5,1,0.3,0.05,0.9,1/50,'birth'],\
                    ]
mean_dices_batch = dict()
for Parameters in ParametersToTest:
    normalize, sigma, radius_ball, zero_boundary, min_dist_bars, ratio_upper, ratio_lower, select_bar = Parameters

    dices_batch = dict()
    for n_image in range(21,38+1):
        IPython.display.clear_output(wait=True)
        print('Parameters =', Parameters)
        print('n_image =', n_image)

        # Open image
        img, seg_medecin = OpenSAT(n_image)

        # Preprocess
        if radius_ball>0:   img = skimage.morphology.erosion(img, footprint=skimage.morphology.ball(radius_ball), out=None, shift_x=False, shift_y=False)
        if sigma>0:         img = scipy.ndimage.gaussian_filter(img, sigma=sigma)    
        if zero_boundary>0: img[np.where(img<zero_boundary)]=1

        # Segmentation
        seg_final, DICEbySLICE = Step2_SAT(img,min_dist_bars=min_dist_bars,ratio_upper=ratio_upper,ratio_lower=ratio_lower,select_bar=select_bar,verbose=verbose,plot=plot)

        # Save dice
        dices_batch[n_image] = DICE(seg_final,seg_medecin, verbose=False)
    
    # Save mean dice
    mean_dices_batch[tuple(Parameters)] = np.mean(list(dices_batch.values()))

In [ ]:
mean_dices_batch

# Segmentation 3D - with diagram

In [ ]:
n_image = 30 #between 21 and 38 included

# Open image
n_image_str = '0'*(3-len(str(n_image)))+str(n_image)
filename = '/home/raph/GoogleDrive/Professionnel/TDA+Brains/Code Python/data/fetal/STA'+str(n_image)+'.nii.gz'
img = nib_load(filename).get_fdata(); print(np.shape(img))
img /= np.max(img)

# Open tissue
n_image_str = '0'*(3-len(str(n_image)))+str(n_image)
filename = '/home/raph/GoogleDrive/Professionnel/TDA+Brains/Code Python/data/fetal/STA'+str(n_image)+'_tissue.nii.gz'
img_tissue = nib_load(filename).get_fdata(); print(np.shape(img))

# Define segmentation cortical plate
img_seg = ((img_regional==112)+(img_regional==113))*1
print('Segmentation size:', np.sum(img_seg))

# Plot slices
z_pos = [70,95,105]
figsize=(len(z_pos)*5,5)
fig = plt.figure(figsize=figsize); 
for i in range(len(z_pos)):
    ax = fig.add_subplot(1,len(z_pos),i+1); ax.axis('off')
    ax.imshow(img[:,:,z_pos[i]],cmap='gray',alpha=0.9,origin='lower'); 
    ax.imshow(img_seg[:,:,z_pos[i]],alpha=0.5,**DLT_KW_SEG); 
    
' Select only one bar '

img_slice = img.copy()
img_slice[np.where(img_slice==0)]=1
img_medecin_slice = img_seg.copy()

homology_dim = 2

# Pre-process
sigma = 0.5
if sigma>0: img_slice = scipy.ndimage.gaussian_filter(img_slice, sigma=sigma)    

# Compute barcode
start_time = ChronometerStart('Compute diagram... ')
barcode = cripser.computePH(img_slice,maxdim=homology_dim) # Compute diagram
ChronometerStop(start_time, method='ms')

# Plot diagram
fig = plt.figure(figsize=(12,4)); ax = fig.add_subplot(1,3,1)
dim_max = int(max([bar[0] for bar in barcode]))
persim.plot_diagrams([np.array([bar[1:3] for bar in barcode if bar[0]==i]) for i in range(dim_max+1)],ax=ax)
ax.set_title('Persistence diagram of flair',fontsize=10);

# Get longest bar
H = [list(bar[1::]) for bar in barcode if bar[0]==homology_dim]          # Allow infinite bars
# H = [bar for _,bar in sorted(zip([bar[1]-bar[0] for bar in H],H))[::-1]] # Sort list H by persistence
H = [bar for _,bar in sorted(zip([bar[0] for bar in H],H))]              # Sort list H by birth

i = 5
bar = H[i]
pos = np.array(bar[2:5]).astype(int)
t = bar[0]+0.0000001
patch = plt.Circle((bar[0],bar[1]), 0.01,fill=False); ax.add_patch(patch)
print('t = ', t, '-', 'value of pixel =', img_slice[pos[0],pos[1],pos[2]])

# Get CC
imt = (img_slice<=t)*1
labels = skimage.measure.label(imt, background=0)
labeltumor = labels[pos[0],pos[1],pos[2]]
CC = (labels == labeltumor)*1
print('Width of CC:', np.sum(CC))

# Plot
ax = fig.add_subplot(1,3,2); ax.axis('off'); ax.set_title('TDA seg');
ax.imshow(img_slice[:,:,100],cmap='gray',alpha=0.5,origin='lower'); 
ax.imshow(CC[:,:,100],alpha=0.5,**DLT_KW_SEG); 

ax = fig.add_subplot(1,3,3); ax.axis('off'); ax.set_title('Groundtruth');
ax.imshow(img_slice[:,:,100],cmap='gray',alpha=0.5,origin='lower'); 
ax.imshow(img_medecin_slice[:,:,100],alpha=0.5,**DLT_KW_SEG);

# Compute dice
dice = DICE(CC,img_medecin_slice, verbose=False)
print('Dice score:', dice)

# II - FeTA dataset

In [ ]:
def OpenFeTA(n_image, verbose=False):
    '''
    i must be between 1 and 50 included 
    /!\ annotations available only up to 40 /!\
    '''
    # Open image
    if n_image <= 9:  filename = '/home/raph/GoogleDrive/Professionnel/TDA+Brains/Code Python/data/fetal_FeTA/sub-feta00'+str(n_image)+'/anat/sub-feta00'+str(n_image)+'_T2w.nii.gz'
    else:             filename = '/home/raph/GoogleDrive/Professionnel/TDA+Brains/Code Python/data/fetal_FeTA/sub-feta0'+str(n_image)+'/anat/sub-feta0'+str(n_image)+'_T2w.nii.gz'
    img = nib_load(filename).get_fdata();
    img /= np.max(img)

    # Open tissue
    if n_image <= 9:  filename = '/home/raph/GoogleDrive/Professionnel/TDA+Brains/Code Python/data/fetal_FeTA/derivatives/sub-feta00'+str(n_image)+'/anat/sub-feta00'+str(n_image)+'_T2w-SR_dseg.nii.gz'
    else:             filename = '/home/raph/GoogleDrive/Professionnel/TDA+Brains/Code Python/data/fetal_FeTA/derivatives/sub-feta0'+str(n_image)+'/anat/sub-feta0'+str(n_image)+'_T2w-SR_dseg.nii.gz'
    img_tissue = nib_load(filename).get_fdata(); 

    # Define segmentation cortical plate
    img_seg = (img_tissue==2)*1
    if verbose: print('Segmentation size:', np.sum(img_seg))

    return img, img_seg

In [ ]:
' Plot only one image '

n_image = 1 #between 1 and 50 included

# Open image
img, img_seg = OpenFeTA(n_image)
print('n_image =', n_image, '/ shape =', np.shape(img))

# Plot slices
z_pos = [70,95,105]
z_pos = [int(i) for i in np.linspace(60,np.shape(img)[2]-50,5)]
figsize=(len(z_pos)*5,5)
fig = plt.figure(figsize=figsize); 
fig.subplots_adjust(wspace=0.05, hspace=0)
for i in range(len(z_pos)):
    ax = fig.add_subplot(1,len(z_pos),i+1); ax.axis('off')
    ax.imshow(img[:,:,z_pos[i]],cmap='gray',alpha=0.9,origin='lower'); 
    ax.imshow(img_seg[:,:,z_pos[i]],alpha=0.5,**DLT_KW_SEG); 
ax.set_title('n_image = '+repr(n_image));

In [ ]:
' Plot all images - y (coronal plane) '

for n_image in range(1,50+1):
    # Open image
    img, img_seg = OpenFeTA(n_image)

    # Plot slices
    z_pos = [int(i) for i in np.linspace(60,np.shape(img)[1]-60,7)]
    figsize=(len(z_pos)*5,5)
    fig = plt.figure(figsize=figsize); 
    fig.subplots_adjust(wspace=0.05, hspace=0)
    for i in range(len(z_pos)):
        ax = fig.add_subplot(1,len(z_pos),i+1); ax.axis('off')
        ax.imshow(img[:,z_pos[i],:],cmap='gray',alpha=0.9,origin='lower'); 
        ax.imshow(img_seg[:,z_pos[i],:],alpha=0.5,**DLT_KW_SEG); 
    ax.set_title('n_image = '+repr(n_image));
    plt.show()

In [ ]:
' Segmentation on one image '


n_image = 2 #between 1 and 50 included

# Parameters
sigma = 0.5                # Preprocess, Gaussian blur
radius_ball = 0          # Step 2, dilation parameter
min_dist_bars = 0.03     # Step 2, to consider multiple bars
zero_boundary = 0.3      # Preprocess, to suppress the boundary
# parameters = [sigma,radius_ball,min_dist_bars,zero_boundary]

verbose, plot = True, True
# verbose, plot = False, False

# Open image
img, seg_medecin = OpenFeTA(n_image)
seg_final = np.zeros(np.shape(img))
print('n_image =', n_image, '/ shape =', np.shape(img))

# Preprocess
if radius_ball>0:   img = skimage.morphology.erosion(img, footprint=skimage.morphology.ball(radius_ball), out=None, shift_x=False, shift_y=False)
if sigma>0:         img = scipy.ndimage.gaussian_filter(img, sigma=sigma)    
if zero_boundary>0: img[np.where(img<zero_boundary)]=1

# Get nonempty slices of im
nonempty_slices = np.where(np.sum(np.sum(img,0),1))[0]
ymin_im, ymax_im = nonempty_slices[0], nonempty_slices[-1]
if verbose: print('ymin ymax =', ymin_im, ymax_im)

if not verbose:
    msg = 'Compute DICEs... '
    start_time = ChronometerStart(msg)
for y in range(ymin_im,ymax_im+1):
# for y in [40]:
    ' Select only one bar '

    # Define slice
    img_slice = img[:,y,:].copy()
    seg_medecin_slice = seg_medecin[:,y,:].copy()

    # Compute barcode
    if verbose: start_time = ChronometerStart(repr(y)+' Compute diagram... ')
    barcode = cripser.computePH(img_slice,maxdim=1) # Compute diagram
    if verbose: ChronometerStop(start_time, method='s')

    # Get longest bar
    homology_dim = 1
    H = [list(bar[1::]) for bar in barcode if bar[0]==homology_dim]          # Allow infinite bars
    H = [bar for _,bar in sorted(zip([bar[1]-bar[0] for bar in H],H))[::-1]] # Sort list H by persistence

    if len(H)>=1:
        # Plot diagram
        if plot:
            fig = plt.figure(figsize=(8,2)); ax = fig.add_subplot(1,3,1)
            dim_max = int(max([bar[0] for bar in barcode]))
            persim.plot_diagrams([np.array([bar[1:3] for bar in barcode if bar[0]==i]) for i in range(dim_max+1)],ax=ax)
            ax.set_title('Persistence diagram of flair',fontsize=10);

        # Get width holes
        WidthHoles = []
        for bar in H:
            # Get CC
            pos, t = np.array(bar[2:4]).astype(int), bar[0]+0.0000001
            labels = skimage.measure.label((img_slice<=t)*1, background=0)
            CC = (labels == labels[pos[0],pos[1]])*1

            # Compute width hole
            labels = skimage.measure.label(1 - CC, background=0)
            components = [(labels==i)*1 for i in range(1,np.max(labels)+1)]
            components_len = [np.sum(component) for component in components]
            width_hole = np.size(CC) - max(components_len) - np.sum(CC)
                # size of the image, minus the backgroud component, minus the CC
                
#             print(width_hole, np.sum(img_slice<1)/10, width_hole + np.sum(CC), np.sum(img_slice<1))
            # Save width hole
            if width_hole + np.sum(CC) < np.sum(img_slice<1) and width_hole > np.sum(img_slice<1)/10:       
                                # The cycle is not the entire boundary, and the width is not too small
                WidthHoles.append(width_hole)
            else:   
                WidthHoles.append(0)

        if any(WidthHoles): # If there is a nonzero value
            # Get optimal bar
            i = np.argsort(WidthHoles)[-1] # Cycle that bounds the largest hole
            bar = H[i]

            # Get CC
            pos, t = np.array(bar[2:4]).astype(int), bar[0]+0.0000001
            imt = (img_slice<=t)*1
            labels = skimage.measure.label(imt, background=0)
            labeltumor = labels[pos[0],pos[1]]
            CC = (labels == labeltumor)*1
            if plot: patch = plt.Circle((bar[0],bar[1]), 0.01,fill=False); ax.add_patch(patch)
#             if verbose: print('t = ', t, '-', 'value of pixel =', img_slice[pos[0],pos[1]])

            # Check if exists a similar bar
            if len(H)>=2:
                distance_bars = np.linalg.norm(np.array(H)[:,0:2]-bar[0:2],axis=1)
                j = np.argsort(distance_bars)[1]
                bar_closest = H[j] # Closest bar
                dist = np.linalg.norm(np.array(bar[0:2])-np.array(bar_closest[0:2]))
                if verbose: 
                    if dist>min_dist_bars:   print('No similar bar:', dist, '>', min_dist_bars)
                    elif dist<=min_dist_bars: print('Exists a similar bar:', dist, '<=', min_dist_bars)
                if dist<=min_dist_bars:
                    # Get CC
                    pos, t = np.array(bar_closest[2:4]).astype(int), bar_closest[0]+0.0000001
                    imt = (img_slice<=t)*1
                    labels = skimage.measure.label(imt, background=0)
                    labeltumor = labels[pos[0],pos[1]]
                    CC_closest = (labels == labeltumor)*1
                    if plot: patch = plt.Circle((bar_closest[0],bar_closest[1]), 0.01,fill=False); ax.add_patch(patch)
    #                 if verbose: print('t = ', t, '-', 'value of pixel =', img_slice[pos[0],pos[1]])

                    # Join CC
                    CC = (CC+CC_closest>0)*1
        else:
            CC = np.zeros(np.shape(img_slice))

        # Plot
        if plot:
            ax = fig.add_subplot(1,3,2); ax.axis('off'); ax.set_title('TDA seg');
            ax.imshow(img_slice,cmap='gray',alpha=0.5,origin='lower'); 
            ax.imshow(CC,alpha=0.5,**DLT_KW_SEG); 

            ax = fig.add_subplot(1,3,3); ax.axis('off'); ax.set_title('Groundtruth');
            ax.imshow(img_slice,cmap='gray',alpha=0.5,origin='lower'); 
            ax.imshow(seg_medecin_slice,alpha=0.5,**DLT_KW_SEG);
            plt.show()

        if verbose:
            # Compute dice
            dice = DICE(CC,seg_medecin_slice, verbose=False)
            print(y, 'Dice score ', round(dice,3), '<------------------------------------------------')

        seg_final[:,y,:] = CC

    if not verbose:    ChronometerTick(start_time, y-ymin_im, ymax_im-ymin_im+1, msg)

    
# Compute dice
dice = DICE(seg_final,seg_medecin, verbose=False)
print('Final Dice score ', round(dice,3))    

In [ ]:
' Batch '

# Parameters
normalize= 'max'         # Preprocess, divide by max or 255
sigma = 0.5                # Preprocess, Gaussian blur
radius_ball = 0          # Step 2, dilation parameter
min_dist_bars = 0.03     # Step 2, to consider multiple bars
zero_boundary = 0.3      # Preprocess, to suppress the boundary
parameters = [normalize,sigma,radius_ball,min_dist_bars,zero_boundary]

verbose, plot = False, False

dices_batch = dict()

for n_image in range(1,40+1):
    # Open image
    img, seg_medecin = OpenFeTA(n_image)
    seg_final = np.zeros(np.shape(img))
    if verbose: print('n_image =', n_image, '/ shape =', np.shape(img))

    # Preprocess
    if radius_ball>0:   img = skimage.morphology.erosion(img, footprint=skimage.morphology.ball(radius_ball), out=None, shift_x=False, shift_y=False)
    if sigma>0:         img = scipy.ndimage.gaussian_filter(img, sigma=sigma)    
    if zero_boundary>0: img[np.where(img<zero_boundary)]=1

    # Get nonempty slices of im
    nonempty_slices = np.where(np.sum(np.sum(img,0),1))[0]
    ymin_im, ymax_im = nonempty_slices[0], nonempty_slices[-1]
    if verbose: print('ymin ymax =', ymin_im, ymax_im)

    if not verbose:
        msg = 'Compute DICEs... '
        start_time = ChronometerStart(msg)
    for y in range(ymin_im,ymax_im+1):
        ' Select only one bar '

        # Define slice
        img_slice = img[:,y,:].copy()
        seg_medecin_slice = seg_medecin[:,y,:].copy()

        # Compute barcode
        if verbose: start_time = ChronometerStart(repr(y)+' Compute diagram... ')
        barcode = cripser.computePH(img_slice,maxdim=1) # Compute diagram
        if verbose: ChronometerStop(start_time, method='s')

        # Get longest bar
        homology_dim = 1
        H = [list(bar[1::]) for bar in barcode if bar[0]==homology_dim]          # Allow infinite bars
        H = [bar for _,bar in sorted(zip([bar[1]-bar[0] for bar in H],H))[::-1]] # Sort list H by persistence

        if len(H)>=1:
            # Plot diagram
            if plot:
                fig = plt.figure(figsize=(8,2)); ax = fig.add_subplot(1,3,1)
                dim_max = int(max([bar[0] for bar in barcode]))
                persim.plot_diagrams([np.array([bar[1:3] for bar in barcode if bar[0]==i]) for i in range(dim_max+1)],ax=ax)
                ax.set_title('Persistence diagram of flair',fontsize=10);

            # Get width holes
            WidthHoles = []
            for bar in H:
                # Get CC
                pos, t = np.array(bar[2:4]).astype(int), bar[0]+0.0000001
                labels = skimage.measure.label((img_slice<=t)*1, background=0)
                CC = (labels == labels[pos[0],pos[1]])*1

                # Compute width hole
                labels = skimage.measure.label(1 - CC, background=0)
                components = [(labels==i)*1 for i in range(1,np.max(labels)+1)]
                components_len = [np.sum(component) for component in components]
                width_hole = np.size(CC) - max(components_len) - np.sum(CC)
                    # size of the image, minus the backgroud component, minus the CC

                # Save width hole
                if width_hole + np.sum(CC) < np.sum(img_slice<1) and width_hole > np.sum(img_slice<1)/10:       
                                    # The cycle is not the entire boundary, and the width is not too small
                    WidthHoles.append(width_hole)
                else:   
                    WidthHoles.append(0)

            if any(WidthHoles): # If there is a nonzero value
                # Get optimal bar
                i = np.argsort(WidthHoles)[-1] # Cycle that bounds the largest hole
                bar = H[i]

                # Get CC
                pos, t = np.array(bar[2:4]).astype(int), bar[0]+0.0000001
                imt = (img_slice<=t)*1
                labels = skimage.measure.label(imt, background=0)
                labeltumor = labels[pos[0],pos[1]]
                CC = (labels == labeltumor)*1
                if plot: patch = plt.Circle((bar[0],bar[1]), 0.01,fill=False); ax.add_patch(patch)

                # Check if exists a similar bar
                if len(H)>=2:
                    distance_bars = np.linalg.norm(np.array(H)[:,0:2]-bar[0:2],axis=1)
                    j = np.argsort(distance_bars)[1]
                    bar_closest = H[j] # Closest bar
                    dist = np.linalg.norm(np.array(bar[0:2])-np.array(bar_closest[0:2]))
                    if verbose: 
                        if dist>min_dist_bars:   print('No similar bar:', dist, '>', min_dist_bars)
                        elif dist<=min_dist_bars: print('Exists a similar bar:', dist, '<=', min_dist_bars)
                    if dist<=min_dist_bars:
                        # Get CC
                        pos, t = np.array(bar_closest[2:4]).astype(int), bar_closest[0]+0.0000001
                        imt = (img_slice<=t)*1
                        labels = skimage.measure.label(imt, background=0)
                        labeltumor = labels[pos[0],pos[1]]
                        CC_closest = (labels == labeltumor)*1
                        if plot: patch = plt.Circle((bar_closest[0],bar_closest[1]), 0.01,fill=False); ax.add_patch(patch)

                        # Join CC
                        CC = (CC+CC_closest>0)*1
            else:
                CC = np.zeros(np.shape(img_slice))

            # Plot
            if plot:
                ax = fig.add_subplot(1,3,2); ax.axis('off'); ax.set_title('TDA seg');
                ax.imshow(img_slice,cmap='gray',alpha=0.5,origin='lower'); 
                ax.imshow(CC,alpha=0.5,**DLT_KW_SEG); 

                ax = fig.add_subplot(1,3,3); ax.axis('off'); ax.set_title('Groundtruth');
                ax.imshow(img_slice,cmap='gray',alpha=0.5,origin='lower'); 
                ax.imshow(seg_medecin_slice,alpha=0.5,**DLT_KW_SEG);
                plt.show()

            if verbose:
                # Compute dice
                dice = DICE(CC,seg_medecin_slice, verbose=False)
                print(y, 'Dice score ', round(dice,3), '<------------------------------------------------')

            seg_final[:,y,:] = CC

        if not verbose: ChronometerTick(start_time, y-ymin_im, ymax_im-ymin_im+1, msg)

    # Compute dice
    dice = DICE(seg_final,seg_medecin, verbose=False)
    dices_batch[n_image] = dice
    print('n_image =', n_image, '- final Dice score ', round(dice,3), '<------------------------------------------------')
    
# Mean dice score
print('Mean dice score:', round(np.mean(list(dices_batch.values())),4), '- parameters', parameters)

# Plot curve
eps = 0.03
plt.figure(figsize=(10,1.5))
plt.plot(dices_batch.keys(),dices_batch.values(), 'black')
plt.scatter(dices_batch.keys(),dices_batch.values(), c='black')
plt.xlim(1-0.5,40+0.5); plt.xticks(range(1,40+1)); 
plt.ylim(min(dices_batch.values())-eps,max(dices_batch.values())+eps); plt.yticks(np.linspace(min(dices_batch.values()),max(dices_batch.values()),5));
plt.title('Dice score per patient');

# Verify model

In [ ]:
' Verify model - homology '

# n_image = 22 #between 21 and 38 included

verbose, plot = False, False

# Parameters
normalize= 'max'           # Preprocess, divide by max or 255
sigma = 0.5                # Preprocess, Gaussian blur
radius_ball = 0            # Preprocess, dilation parameter
# zero_boundary = 0.3        # Preprocess, to suppress the boundary
min_dist_bars = 0.03       # Step 2, to consider multiple bars
ratio_upper = 3/4          # Step 2, max relative size of object (circle)
ratio_lower = 1/50         # Step 2, min relative size of interior
select_bar = 'birth'       # Step 2, optimal bar according to 'width' (of hole), 'pers' (of bar) or 'birth' 
Parameters = [normalize,sigma,radius_ball,zero_boundary,min_dist_bars,ratio_upper,ratio_lower,select_bar]

mean_bool_CC = dict()
max_number_CC = dict()

msg = 'Verify model... '
start_time = ChronometerStart(msg)
for n_image in range(21,38+1):
# for n_image in [26]:
    # Open image
    img, seg_medecin = OpenSAT(n_image)

    # Preprocess
    if radius_ball>0:   img = skimage.morphology.erosion(img, footprint=skimage.morphology.ball(radius_ball), out=None, shift_x=False, shift_y=False)
    if sigma>0:         img = scipy.ndimage.gaussian_filter(img, sigma=sigma)    
#     if zero_boundary>0: img[np.where(img<zero_boundary)]=1

    ' Verif model 2D '
    
    # Get nonempty slices of im
    nonempty_slices = np.where(np.sum(np.sum(seg_medecin,0),1))[0]
    ymin_im, ymax_im = nonempty_slices[0], nonempty_slices[-1]
    y_im = range(ymin_im,ymax_im+1)
    if verbose: print('ymin ymax =', ymin_im, ymax_im)

    components_number_byslice = dict()
    for y in y_im:
        # Define slice
        img_slice = img[:,y,:].copy()
        seg_medecin_slice = seg_medecin[:,y,:].copy()
        if radius_ball>0: 
            seg_medecin_slice = skimage.morphology.binary_dilation(seg_medecin_slice, footprint=skimage.morphology.disk(radius_ball))

        # Extract CC
        seg_complement = 1 - seg_medecin_slice
        labels = skimage.measure.label(seg_complement, background=0)
        components = [(labels==i)*1 for i in range(1,np.max(labels)+1)] # Remove cortical plate

#         ratio_big_component = 0.1
#         components_number_byslice[y] = len([component for component in components\
#                                             if np.sum(component)>np.sum(img_slice>0)*ratio_big_component])
        ratio_big_component = 0.01
        components_number_byslice[y] = len([component for component in components\
                                            if np.sum(component)>np.prod(np.shape(img_slice))*ratio_big_component])

    mean_bool_CC[n_image]  = np.mean([components_number_byslice[y] in [2,3] for y in components_number_byslice])
    max_number_CC[n_image] = max(components_number_byslice.values())
        
    ChronometerTick(start_time, n_image, 38, msg)  
print('Per image, mean number of slices satisfying (H2\'):', repr(round(np.mean(list(mean_bool_CC.values()))*100,2))+'%')
print('Max number of CC in all slice:', max(max_number_CC.values()))

In [ ]:
' Plot consecutive slices '

n_image = 38
img, seg_medecin = OpenSAT(n_image)
y_values = [25, 60, 100, 130, 153]

fig = plt.figure(figsize=(15,3)); fig.subplots_adjust(wspace=0.05, hspace=0.05)
for i, y in enumerate(y_values):
    # Define slice
    img_slice = img[:,y,:].copy()
    seg_medecin_slice = seg_medecin[:,y,:].copy()
    
    ax = fig.add_subplot(1,len(y_values),i+1); ax.axis('off')
    n = 10
    ax.imshow(img_slice[:,n:-n].T,cmap='gray',alpha=0.6,origin='lower'); 
    CC = seg_medecin_slice[:,n:-n].T
    ax.imshow(CC,alpha=1,**DLT_KW_SEG); 

plt.savefig('Images/fetal_consecutive_slices.pdf', format="pdf", bbox_inches="tight")